# Multi-Model Image-Text Search

## 1. Import all requirements

In [1]:
import kagglehub
import pandas as pd
import numpy as np
import os
import re
import math

# Scikit-Learn
from sklearn.model_selection import train_test_split

# PyTorch
import torch, open_clip
import torchvision.transforms as T
from torch.optim import AdamW
from torch.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader

from PIL import Image
import faiss
import gradio as gr

/home/mark/Documents/python/projects/CLIP_fashion/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Download the dataset

In [ ]:
# Dowloading the dataset (in this repo is already installed in data)
path = kagglehub.dataset_download("paramaggarwal/fashion-product-images-small")

/home/mark/Documents/python/projects/CLIP_fashion/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 565M/565M [01:31<00:00, 6.50MB/s] 


Extracting files...


## 3. Preprocess the dataset

In [ ]:
# Get corrected styles.csv
styles_path = './data/styles.csv'
df = pd.read_csv(styles_path)


### ----- IMAGE PATH PREPROCESS -----

# Add new column 'image' reffers to the full path to the image correlated to 'id'
img_root = './data/images'
df['image'] = df['id'].astype(str).radd(img_root + '/') + '.jpg'

# Check all paths are exist and drop all which doesn't exist
df = df[df['image'].map(os.path.exists)].reset_index(drop=True)


### ----- TEXT PREPROCESS -----

# Shorten, remove unnecessary, clean, process and polish text
cols = ['productDisplayName', 'masterCategory', 'gender', 'baseColour', 'usage', 'year']
df = df[cols + ['id', 'image']].dropna(subset=['productDisplayName', 'masterCategory'])

# Create a normalization function for text to remove unnecessary
def norm(s):
    s = str(s).lower()
    s = re.sub(r'[_/|]+', ' ', s)
    s = re.sub(r'\s+', ' ', re.sub(r'[^a-z0-9\s,-]', '', s)).strip()
    return s

# Apply normalization and merge all important columns in 'text'
df['text'] = (
    df['productDisplayName'].map(norm) + ', ' +
    df['gender'].map(norm) + ', ' +
    df['masterCategory'].map(norm) + ', ' +
    df['baseColour'].map(norm) + ', ' +
    df['usage'].map(norm) + ', ' +
    df['year'].astype(str)
).str.replace(r'\s+,', ',', regex=True).str.replace(r',\s*,', ',', regex=True)

# Keep text short up to 40
df['text'] = df['text'].str.split().str[:40].str.join(' ')

# Drop all used columns (they won't be used further)
dataset = df[['image', 'text']]


### ----- SPLIT -----

# Train/Val/Test split
train, tmp = train_test_split(dataset, test_size=0.2, random_state=42)
val, test = train_test_split(tmp, test_size=0.5, random_state=42)

# Save validation dataset for app usage
val.to_scv('./data/val.csv')

print(f'Train size: {len(train)} examples\nVal size: {len(val)} examples\nTest size: {len(test)} examples')

Train size: 35547 examples
Val size: 4443 examples
Test size: 4444 examples


## 4. Zero-shot CLIP Baseline Metrics

### 4.1 Dataloader & Model

In [ ]:
# Set the device
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Download model and tokenizer
model, _, preprocess = open_clip.create_model_and_transforms(
  'ViT-B-32', pretrained='laion2b_s34b_b79k', device=device
)

tokenizer = open_clip.get_tokenizer('ViT-B-32')

# Create pair dataset class
class PairDS(Dataset):
  def __init__(self, frame):
    self.f = frame.reset_index(drop=True)

  def __len__(self): return len(self.f)

  def __getitem__(self, i):
    img = preprocess(Image.open(self.f.image[i]).convert('RGB'))
    txt = self.f.text[i]

    return img, txt


BATCH = 128

# Take up to 4k example in validation
val_ds = PairDS(val.sample(min(len(val), 4000), random_state=0))
val_dl = DataLoader(val_ds, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

### 4.2 Encode images and texts

In [ ]:
@torch.no_grad()
def encode_split(dloader):
  model.eval()
  img_embs, txt_embs = [], []

  for imgs, txts in dloader:
    imgs = imgs.to(device)

    # Tokenize text
    txt_tokens = tokenizer(list(txts)).to(device)

    # Encode data
    img_feat = model.encode_image(imgs)
    txt_feat = model.encode_text(txt_tokens)

    # Normalize data for 'Cosine Similarity'
    img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True)
    txt_feat = txt_feat / txt_feat.norm(dim=-1, keepdim=True)
    img_embs.append(img_feat.cpu())
    txt_embs.append(txt_feat.cpu())

  return torch.cat(img_embs), torch.cat(txt_embs)

# Run encoding data
img_Z, txt_Z = encode_split(val_dl)

img_Z.shape, txt_Z.shape

(torch.Size([4000, 512]), torch.Size([4000, 512]))

### 4.3 Metrics: Recall@K & nDCG@K

In [ ]:
# cosine via normalized embeddings → inner product
def build_index(X):
  X = X.numpy().astype('float32')

  index = faiss.IndexFlatIP(X.shape[1])
  index.add(X)

  return index

# Metrics Recall@K & nDCG
def recall_ndcg(query, key_Z, K=10):
  index = build_index(key_Z)

  D, I = index.search(query.numpy().astype('float32'), K) # Top K indecis
  N = I.shape[0]

  # Hits at K
  ranks = np.array([np.where(I[i] == i)[0][0] + 1 if i in I[i] else np.inf for i in range(N)])
  recall_at = {k: float(np.mean(ranks <= k)) for k in [1, 5, 10]}

  # nDCG@K (relevance 1 at true match, else 0)
  def dcg_at_k(idx_row):
    return 1.0 / np.log2(np.where(idx_row == np.arange(len(idx_row))[:, None][1] + 2)) if False else None

  # simple nDCG@K since single relevant item:
  ndcg_at = {}

  for k in [1, 5, 10]:
    hits = (I[:, :k] == np.arange(N)[:, None]).astype(float)
    # DCG = 1/log2(rank+1) if hit else 0; IDCG = 1
    gains = hits / np.log2(np.arange(2, k+2))
    ndcg_at[k] = float(gains.max(axis=1).mean())

  return recall_at, ndcg_at


### ----- VAL AT THE BASELINE -----

# text → image
bl_r_t2i, bl_n_t2i = recall_ndcg(txt_Z, img_Z, K=10)
# image → text
bl_r_i2t, bl_n_i2t = recall_ndcg(img_Z, txt_Z, K=10)

print('Val Text→Image:')
print(f'Recall@1/5/10: {bl_r_t2i}')
print(f'nDCG:          {bl_n_t2i}\n')

print('Val Image→Text:')
print(f'Recall@1/5/10: {bl_r_i2t}')
print(f'nDCG:          {bl_n_i2t}\n')

Val Text→Image:
Recall@1/5/10: {1: 0.231, 5: 0.51, 10: 0.65975}
nDCG:          {1: 0.231, 5: 0.3755963603221858, 10: 0.4241229663194789}

Val Image→Text:
Recall@1/5/10: {1: 0.2465, 5: 0.547, 10: 0.69725}
nDCG:          {1: 0.2465, 5: 0.4010583939221364, 10: 0.4495314896506227}



## 5. Lightweight Fine-Tuning

### 5.1 Models, Dataset & Loaders

In [ ]:
# image augs (train stronger, val just center-crop/normalize)
model, _, preprocess_val = open_clip.create_model_and_transforms(
  'ViT-B-32', pretrained='laion2b_s34b_b79k', device=device
)

tokenizer = open_clip.get_tokenizer('ViT-B-32')

# Build a "train" augmentation: start with preprocess_val but add jitter
train_tf = T.Compose([
    T.Resize((preprocess_val.transforms[0].size, preprocess_val.transforms[0].size)),
    T.RandomResizedCrop(preprocess_val.transforms[1].size, scale=(0.9, 1.0)),
    T.ColorJitter(0.1,0.1,0.1,0.05),
    T.RandomHorizontalFlip(p=0.5),
    *preprocess_val.transforms[2:]  # to_tensor + normalize
])


class PairDBTransforms(Dataset):
  def __init__(self, df, tf):
    self.df = df.reset_index(drop=True)
    self.tf = tf

  def __len__(self):
    return len(self.df)

  def __getitem__(self, i):
    x = Image.open(df.image[i]).convert('RGB')

    return self.tf(x), self.df.text[i]


# Create transformed datasets
train_ds = PairDBTransforms(train, train_tf)
val_ds = PairDBTransforms(val, preprocess_val)

def collate(batch):
  imgs, txts = zip(*batch)
  return torch.stack(imgs), list(txts)


BATCH = 512

# Create DataLoaders
train_dl = DataLoader(train_ds, batch_size=BATCH, num_workers=2, shuffle=True, pin_memory=True, collate_fn=collate, drop_last=True)
val_dl = DataLoader(val_ds, batch_size=BATCH, num_workers=2, shuffle=False, pin_memory=True, collate_fn=collate)

### 5.2 Freeze policy

In [ ]:
# Last N transformer blocks to unfreeze
N = 1

# Unfreeze text/image projection + last N blocks; freeze rest
for p in model.parameters(): p.requires_grad = False

# Text side
for p in model.text_projection: p.requires_grad = True
for p in model.transformer.resblocks[-N:].parameters(): p.requires_grad = True

# Vision side
for p in model.visual.proj: p.requires_grad = True
for p in model.visual.transformer.resblocks[-N:].parameters(): p.requires_grad = True


# Show all trainable parameters
trainable_params = [p for p in model.parameters() if p.requires_grad]
n_trainable_p = sum(p.numel() for p in trainable_params) / 1e6

print(f'Total trainable params: {n_trainable_p:.1f} million')

# Show total amount of parameters
n_total_p = sum(p.numel() for p in model.parameters()) / 1e6

print(f'Total model`s params: {n_total_p:,.1f} millon')

# Show % of trainable parameters which will be trained
print(f'{(n_trainable_p / n_total_p) * 100:.2f}% of params will be trained')

Total trainable params: 30.7 million
Total model`s params: 151.3 millon
20.31% of params will be trained


### 5.3 Loss, optimizer, schedule

In [ ]:
# Hyperparameters
EPOCHS = 8
LR = 3e-5
WARMUP_STEPS = 200
MAX_GRAD = 1.0 # For gradient clipping
total_steps = EPOCHS * len(train_dl)
global_step = 0
best_r1 = 0.0
best_path = './models/clip_ft_best.pt'

# Create optimizer and scaler
opt = AdamW(trainable_params, lr=LR, weight_decay=0.02)
scaler = GradScaler(enabled=(device=='cuda'))

# learnable temperature (logit_scale) already in CLIP; constrain its range
def clamp_logit_scale():
  with torch.no_grad():
    model.logit_scale.clamp_(0, math.log(100)) # exp(logit_scale) in [1, 100]

/tmp/ipython-input-2077855214.py:10: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=(device=='cuda'))


### 5.4 Pre-Training loop process (InfoNCE in both directions)

In [ ]:
# Implement the cosine similarity
def cosine_sim(a, b):
  a = a / a.norm(dim=-1, keepdim=True)
  b = b / b.norm(dim=-1, keepdim=True)

  return (a @ b.T)

def step_schedule(step):
  # simple warmup → cosine decay
  if step < WARMUP_STEPS:
    return step / max(1, WARMUP_STEPS)

  t = (step - WARMUP_STEPS) / max(1, total_steps - WARMUP_STEPS)

  return 0.5 * (1 + math.cos(math.pi * t))

def val_metrics():
  img_Z, txt_Z = encode_split(val_dl)

 ## ----- TEXT -> IMAGE -----
  # Build the text index
  idx = build_index(img_Z)

  D, I = idx.search(txt_Z.numpy().astype('float32'), 10)
  N = I.shape[0]
  ranks = np.array([np.where(I[i]==i)[0][0]+1 if i in I[i] else np.inf for i in range(N)])

  rec_t = {k: float(np.mean(ranks <= k)) for k in [1, 5, 10]}
  ndcg_t = {k: float(((I[:,:k]==np.arange(N)[:,None]).astype(float) / np.log2(np.arange(2,k+2))).max(axis=1).mean()) for k in [1,5,10]}

  ### ----- IMAGE -> TEXT -----
  # Build the image index
  idx = build_index(img_Z)

  D, I = idx.search(img_Z.numpy().astype('float32'), 10)
  ranks = np.array([np.where(I[i]==i)[0][0]+1 if i in I[i] else np.inf for i in range(N)])

  rec_i = {k: float(np.mean(ranks <= k)) for k in [1, 5, 10]}
  ndcg_i = {k: float(((I[:,:k]==np.arange(N)[:,None]).astype(float) / np.log2(np.arange(2,k+2))).max(axis=1).mean()) for k in [1,5,10]}

  return rec_t, ndcg_t, rec_i, ndcg_i

### 5.5 Training loop

In [ ]:
for epoch in range(1, EPOCHS + 1):
  model.train()

  for imgs, txts in train_dl:
    imgs = imgs.to(device)
    tokens = tokenizer(txts).to(device)

    opt.zero_grad(set_to_none=True)

    with autocast(enabled=(device == 'cuda')):
      # Encode
      img = model.encode_image(imgs)
      txt = model.encode_text(tokens)

      # Multiply logit scale by cosine similarity
      logit_scale = model.logit_scale.exp()
      logits_i2t = logit_scale * cosine_sim(img, txt) # I to T
      logits_t2i = logit_scale * cosine_sim(txt, img) # T to I

      # Calculate loss
      targets = torch.arange(len(imgs), device=device)
      loss_i2t = torch.nn.functional.cross_entropy(logits_i2t, targets)
      loss_t2i = torch.nn.functional.cross_entropy(logits_t2i, targets)
      loss = (loss_i2t + loss_t2i) / 2

    # Backpropagate the loss
    scaler.scale(loss).backward()
    torch.nn.utils.clip_grad_norm_(trainable_params, MAX_GRAD)
    scaler.step(opt)
    scaler.update()
    clamp_logit_scale()

    # Manual LR schedule
    for pg in opt.param_groups:
      pg['lr'] = LR * step_schedule(global_step)

    global_step += 1

  # Evaluate each epoch
  r_t, n_t, r_i, n_i = val_metrics()

  r1  = 0.5 * (r_t[1]  + r_i[1])
  r5  = 0.5 * (r_t[5]  + r_i[5])
  r10 = 0.5 * (r_t[10] + r_i[10])

  print(f"Epoch [{epoch}/{EPOCHS}]")
  print(f"T->I   R@1={r_t[1]:.3f} R@5={r_t[5]:.3f} R@10={r_t[10]:.3f}")
  print(f"I->T   R@1={r_i[1]:.3f} R@5={r_i[5]:.3f} R@10={r_i[10]:.3f}")
  print(f"AVG    R@1={r1:.3f}     R@5={r5:.3f}     R@10={r10:.3f}")

  # If avg recall is better than the best save the model
  if r1 > best_r1:
    best_r1 = r1

# Save the model after training
torch.save({'model': model.state_dict()}, best_path)
print(f'Best new path: {best_path}')

/tmp/ipython-input-1451184076.py:10: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(device == 'cuda')):


Epoch [1/8]
T->I   R@1=0.254 R@5=0.550 R@10=0.700
I->T   R@1=0.261 R@5=0.561 R@10=0.713
AVG    R@1=0.257     R@5=0.556     R@10=0.707
Epoch [2/8]
T->I   R@1=0.271 R@5=0.583 R@10=0.743
I->T   R@1=0.280 R@5=0.601 R@10=0.756
AVG    R@1=0.276     R@5=0.592     R@10=0.749
Epoch [3/8]
T->I   R@1=0.289 R@5=0.625 R@10=0.789
I->T   R@1=0.296 R@5=0.642 R@10=0.796
AVG    R@1=0.292     R@5=0.633     R@10=0.793
Epoch [4/8]
T->I   R@1=0.302 R@5=0.671 R@10=0.829
I->T   R@1=0.319 R@5=0.685 R@10=0.840
AVG    R@1=0.311     R@5=0.678     R@10=0.835
Epoch [5/8]
T->I   R@1=0.312 R@5=0.713 R@10=0.871
I->T   R@1=0.339 R@5=0.731 R@10=0.880
AVG    R@1=0.325     R@5=0.722     R@10=0.875
Epoch [6/8]
T->I   R@1=0.333 R@5=0.751 R@10=0.891
I->T   R@1=0.363 R@5=0.768 R@10=0.904
AVG    R@1=0.348     R@5=0.760     R@10=0.898
Epoch [7/8]
T->I   R@1=0.340 R@5=0.770 R@10=0.908
I->T   R@1=0.382 R@5=0.791 R@10=0.917
AVG    R@1=0.361     R@5=0.781     R@10=0.913
Epoch [8/8]
T->I   R@1=0.354 R@5=0.784 R@10=0.913
I->T   R@1=0

### 5.6 Load best for next steps

In [ ]:
ckpt = torch.load('./models/clip_ft_best.pt', map_location=device)
model.load_state_dict(ckpt['model'])
model.eval()

CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
    (patch_dropout): Identity()
    (ln_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): ModuleList(
        (0-11): 12 x ResidualAttentionBlock(
          (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (ls_1): Identity()
          (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=768, out_features=3072, bias=True)
            (gelu): GELU(approximate='none')
            (c_proj): Linear(in_features=3072, out_features=768, bias=True)
          )
          (ls_2): Identity()
        )
      )
    )
    (ln_post): LayerNorm((768,), eps=1e-05, elementwise_affine

## 6. Gradio Application

### 6.1 Build FAISS indeces for both directions

In [ ]:
# ---- Build HNSW index ----
def build_hnsw(vectors, m=32, ef_search=64):
    dim = vectors.shape[1]
    index = faiss.IndexHNSWFlat(dim, m)  # m = neighbors per node
    index.hnsw.efSearch = ef_search
    index.add(vectors.astype('float32'))

    return index

# Encode all (with fine-tuned model)
@torch.no_grad()
def encode_all(df):
    img_embs, txt_embs = [], []

    for img_path, text in zip(df.image, df.text):
        # Preprocess images and tokenize texts
        img = preprocess_val(Image.open(img_path).convert('RGB')).unsqueeze(0).to(device)
        txt = tokenizer([text]).to(device)

        img_vec = model.encode_image(img)
        txt_vec = model.encode_text(txt)

        img_vec /= img_vec.norm(dim=-1, keepdim=True)
        txt_vec /= txt_vec.norm(dim=-1, keepdim=True)

        img_embs.append(img_vec.cpu())
        txt_embs.append(txt_vec.cpu())

    return torch.cat(img_embs).numpy(), torch.cat(txt_embs).numpy()

img_vectors, txt_vectors = encode_all(val)
index_img = build_hnsw(img_vectors)
index_txt = build_hnsw(txt_vectors)

# ---- Search functions ----
def search_by_text(query, k=5):
    with torch.no_grad():
        # Tokenize and encode text
        q_tokens = tokenizer([query]).to(device)
        q_vec = model.encode_text(q_tokens)

        # Normalize
        q_vec /= q_vec.norm(dim=-1, keepdim=True)

        # Search
        D, I = index_img.search(q_vec.cpu().numpy().astype('float32'), k)

    results = []
    for score, idx in zip(D[0], I[0]):
        # Take best images' paths by ids
        img_path = val.image.iloc[idx]

        try:
            # Open an image, make a caption and append in list best results
            img_obj = Image.open(img_path).convert('RGB')
            caption = f"{val.text.iloc[idx]} (score={score:.3f})"
            results.append((img_obj, caption))
        except:
            continue

    return results

def search_by_image(img, k=5):
    with torch.no_grad():
        # Preprocess and encode image
        img_tensor = preprocess_val(img).unsqueeze(0).to(device)
        q_vec = model.encode_image(img_tensor)

        # Normalize
        q_vec /= q_vec.norm(dim=-1, keepdim=True)

        # Search
        D, I = index_txt.search(q_vec.cpu().numpy().astype('float32'), k)

    captions = [f"{val.text.iloc[idx]} (score={score:.3f})" for score, idx in zip(D[0], I[0])]

    return "\n".join(captions)

### 6.2 Gradio UI

In [ ]:
with gr.Blocks() as demo:
    gr.Markdown('## 🖼️ Text ↔ Image Search (CLIP + FAISS HNSW)')
    with gr.Tab('Text → Images'):
        with gr.Row():
            text_in = gr.Textbox(label='Search text', scale=3)
            k_in = gr.Slider(1, 20, value=5, step=1, label='Top‑K', scale=1)
        gallery_out = gr.Gallery(label='Top Matches', columns=5, height='auto')
        dl_btn = gr.Button('Download Top‑K as ZIP')
        dl_file = gr.File(label='Your ZIP will appear here')

        # search on Enter
        text_in.submit(fn=search_by_text, inputs=[text_in, k_in], outputs=gallery_out)
        # also allow click to search
        k_in.change(fn=search_by_text, inputs=[text_in, k_in], outputs=gallery_out)

    with gr.Tab('Image → Text'):
        img_in = gr.Image(label='Upload image', type='pil')
        k2_in = gr.Slider(1, 20, value=5, step=1, label='Top‑K')
        text_out = gr.Textbox(label='Top Captions + Scores')
        img_in.change(fn=search_by_image, inputs=[img_in, k2_in], outputs=text_out)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ba5937ef6f06f41610.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
